In [21]:
import math
import os

import requests
import pandas as pd
from typing import List
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-US,en;q=0.9",
    "cache-control": "no-cache",
    "pragma": "no-cache",
    "origin": "https://www.nike.com",
    "referer": "https://www.nike.com/",
    "sec-ch-ua": '"Chromium";v="127", "Not;A=Brand";v="24", "Google Chrome";v="127"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "cross-site",
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/127.0.0.0 Safari/537.36"
    ),
}


def create_session():
    session = requests.Session()
    
    retry_strategy = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"]
    )
    
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    
    return session


def fetch_turnto_reviews(
    model_code: str,
    site_id: str = "hwf2R4pYhl2NYnKsite",
    country: str = "us",
    locale: str = "en_US",
    limit: int = 20,
    pause: float = 1.0,
) -> List[dict]:
    base_url = f"https://cdn-ws.turnto.com/v5/sitedata/{site_id}/{model_code}/{country}/review/{locale}"
    
    session = create_session()
    
    try:
        print(f"Lade Bewertungen für SKU {model_code}...")
        
        offset = 0
        filters = "{}"
        sort = "LOCAL"
        include_syndicated = "true"
        include_local = "false"
        
        url = f"{base_url}/{offset}/{limit}/{filters}/{sort}/{include_syndicated}/{include_local}/"
        
        r = session.get(url, headers=HEADERS, timeout=(30, 60))
        r.raise_for_status()
        data = r.json()
        
        total = data.get("total", 0)
        pages = math.ceil(total / limit) if total > 0 else 0
        
        print(f"Gefunden: {total} Bewertungen auf {pages} Seiten")
        
        all_rows = []
        for page in range(pages):
            try:
                if page > 0:
                    offset = page * limit
                    url = f"{base_url}/{offset}/{limit}/{filters}/{sort}/{include_syndicated}/{include_local}/"
                    print(f"Lade Seite {page + 1} von {pages}...")
                    time.sleep(pause)
                    
                    r = session.get(url, headers=HEADERS, timeout=(30, 60))
                    r.raise_for_status()
                    data = r.json()

                for rev in data.get("reviews", []):
                    if not str(rev.get("locale", "")).lower().startswith("en"):
                        continue
                    user = rev.get("user", {})
                    cat_item = rev.get("catItem", {})
                    profile_attrs = rev.get("profileAttributes", {})
                    
                    dimensions = {}
                    for dim in rev.get("dimensions", []):
                        dim_label = dim.get("dimensionLabel", "")
                        dim_value = dim.get("value")
                        dim_value_labels = dim.get("valueLabels", [])
                        if dim_value is not None and dim_value < len(dim_value_labels):
                            dimensions[dim_label] = dim_value_labels[dim_value]
                        else:
                            dimensions[dim_label] = dim_value
                    
                    profile_size = None
                    for custom_attr in profile_attrs.get("custom", []):
                        if custom_attr.get("label") == "Deine übliche Größe":
                            profile_size = custom_attr.get("value")
                            break
                    
                    text = rev.get("text", "")
                    text = text.replace("<br />", " ").replace("<br>", " ").replace("\n", " ")
                    
                    all_rows.append({
                        "id": rev.get("id"),
                        "date": rev.get("dateCreatedFormatted"),
                        "date_millis": rev.get("dateCreatedMillis"),
                        "rating": rev.get("rating"),
                        "title": rev.get("title"),
                        "text": text,
                        "recommended": rev.get("recommended"),
                        "up_votes": rev.get("upVotes", 0),
                        "down_votes": rev.get("downVotes", 0),
                        "user_id": user.get("id"),
                        "user_first_name": user.get("firstName"),
                        "user_last_name": user.get("lastName"),
                        "user_nickname": user.get("nickName"),
                        "product_sku": cat_item.get("sku"),
                        "product_title": cat_item.get("title"),
                        "product_url": cat_item.get("url"),
                        "locale": rev.get("locale"),
                        "incentivized": rev.get("incentivized", False),
                        "dimension_fit": dimensions.get("Wie ist die Passform dieses Produkts?"),
                        "dimension_comfort": dimensions.get("Wie bequem war dieses Produkt?"),
                        "dimension_recommend": dimensions.get("Würdest du dieses Produkt weiterempfehlen?"),
                        "profile_location": profile_attrs.get("location"),
                        "profile_size": profile_size,
                    })
                    
                    progress = len(all_rows) / total * 100 if total > 0 else 0
                    print(f"Review {len(all_rows)} von {total} (Seite {page + 1}/{pages}) - {progress:.1f}%")
                
                print(f"Seite {page + 1} erfolgreich geladen - {len(data.get('reviews', []))} Bewertungen")
                
            except requests.exceptions.Timeout as e:
                print(f"Timeout bei Seite {page + 1}: {e}")
                print("Versuche es erneut...")
                time.sleep(5)
                continue
            except requests.exceptions.RequestException as e:
                print(f"Fehler bei Seite {page + 1}: {e}")
                print("Überspringe diese Seite...")
                continue
        
        print(f"Alle Bewertungen erfolgreich geladen: {len(all_rows)} von {total}")
        return all_rows
        
    except requests.exceptions.RequestException as e:
        print(f"Fehler beim Abrufen der Bewertungen: {e}")
        return []
    finally:
        session.close()


def main():
    try:
        print("Starte TurnTo Review Scraper für Nike Air Jordan 1 Low...")
        sku = "HQ7540"
        data = fetch_turnto_reviews(sku)
        
        if data:
            df = pd.DataFrame(data)
            os.makedirs("data", exist_ok=True)
            filename = os.path.join("data", f"nike_reviews_{sku}.csv")
            df.to_csv(filename, index=False, encoding='utf-8')
            print(f"Bewertungen erfolgreich in {filename} gespeichert!")
            print(f"Anzahl Bewertungen: {len(data)}")
        else:
            print("Keine Bewertungen gefunden oder Fehler aufgetreten.")
            
    except Exception as e:
        print(f"Unerwarteter Fehler: {e}")


if __name__ == "__main__":
    main()



Starte TurnTo Review Scraper für Nike Air Jordan 1 Low...
Lade Bewertungen für SKU HQ7540...
Gefunden: 34 Bewertungen auf 2 Seiten
Review 1 von 34 (Seite 1/2) - 2.9%
Review 2 von 34 (Seite 1/2) - 5.9%
Review 3 von 34 (Seite 1/2) - 8.8%
Review 4 von 34 (Seite 1/2) - 11.8%
Review 5 von 34 (Seite 1/2) - 14.7%
Review 6 von 34 (Seite 1/2) - 17.6%
Review 7 von 34 (Seite 1/2) - 20.6%
Review 8 von 34 (Seite 1/2) - 23.5%
Review 9 von 34 (Seite 1/2) - 26.5%
Review 10 von 34 (Seite 1/2) - 29.4%
Review 11 von 34 (Seite 1/2) - 32.4%
Review 12 von 34 (Seite 1/2) - 35.3%
Review 13 von 34 (Seite 1/2) - 38.2%
Review 14 von 34 (Seite 1/2) - 41.2%
Review 15 von 34 (Seite 1/2) - 44.1%
Review 16 von 34 (Seite 1/2) - 47.1%
Review 17 von 34 (Seite 1/2) - 50.0%
Review 18 von 34 (Seite 1/2) - 52.9%
Review 19 von 34 (Seite 1/2) - 55.9%
Review 20 von 34 (Seite 1/2) - 58.8%
Seite 1 erfolgreich geladen - 20 Bewertungen
Lade Seite 2 von 2...
Review 21 von 34 (Seite 2/2) - 61.8%
Review 22 von 34 (Seite 2/2) - 64.7%
